In [9]:
import pandas as pd
from pathlib import Path
import re
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

In [10]:
df = pd.read_csv(r"../data/raw/jobs.csv")
df.shape

(85470, 11)

In [11]:
df.columns.tolist()

['job_title',
 'job_type',
 'position_level',
 'city',
 'experience',
 'skills',
 'job_fields',
 'salary',
 'salary_min',
 'salary_max',
 'unit']

In [ ]:
df.head(10)

In [13]:
df.isnull().sum()

job_title             7
job_type              0
position_level        0
city                 32
experience            0
skills            11294
job_fields         7625
salary                0
salary_min            0
salary_max            0
unit                  0
dtype: int64

In [14]:
df["city"].value_counts().head(10)

city
hà nội         26224
hồ chí minh    25958
bình dương      3722
đồng nai        2324
hải phòng       1663
long an         1612
đà nẵng         1386
hưng yên        1119
bắc ninh        1042
tây ninh         696
Name: count, dtype: int64

In [15]:
keywords = ["lập trình", "developer", "kỹ sư phần mềm", "software",
            "data", "devops", "tester", "backend", "frontend", "it "]
pattern = "|".join(keywords)
it_df = df[df["job_title"].str.contains(pattern, case=False, na=False)]
print(len(it_df))

1618


In [17]:
fields = df["job_fields"].dropna().str.split(",").explode().str.strip()
keyword_fields = r"\bit\b|phần mềm|phần cứng|công nghệ|máy tính|mạng|dữ liệu|lập trình"
fields[fields.str.contains(keyword_fields, case=False, na=False)].value_counts()

job_fields
it phần mềm                         1768
cntt - phần mềm                      817
it phần cứng                         492
cntt - phần cứng                     431
mạng                                 431
it phần cứng - mạng                  288
công nghệ thực phẩm                  256
công nghệ sinh học                   199
công nghệ thực phẩm - dinh dưỡng     102
phân tích - thống kê dữ liệu          50
Name: count, dtype: int64

In [18]:
it_fields = ["it phần mềm", "it phần cứng"]
mask_field = df["job_fields"].str.contains("|".join(it_fields), case=False, na=False)
mask_title = df["job_title"].str.contains(pattern, case=False, na=False)

print("theo job_fields:", mask_field.sum())
print("theo job_title:", mask_title.sum())
print("cả hai:", (mask_field & mask_title).sum())

theo job_fields: 2160
theo job_title: 1618
cả hai: 733


In [19]:
IT_FIELDS = {
    "it phần mềm", "cntt - phần mềm",
    "it phần cứng", "cntt - phần cứng", "it phần cứng - mạng",
    "mạng",
}
DATA_FIELDS = {"phân tích - thống kê dữ liệu"}

df["field_list"] = (
    df["job_fields"].fillna("").str.split(",")
      .apply(lambda xs: [x.strip() for x in xs if x.strip()])
)

mask_it = df["field_list"].apply(lambda xs: any(x in IT_FIELDS for x in xs))
mask_data = df["field_list"].apply(lambda xs: any(x in DATA_FIELDS for x in xs))

print("IT:", mask_it.sum())
print("data analyst:", mask_data.sum(), "| trong đó trùng với IT:", (mask_it & mask_data).sum())

IT: 3111
data analyst: 50 | trong đó trùng với IT: 13


In [20]:
more = r"cntt|an ninh|bảo mật|game|kiểm thử|\bai\b|khoa học dữ liệu|thiết kế web|\bdata\b|điện toán|hệ thống"
fields[fields.str.contains(more, case=False, na=False)].value_counts()

job_fields
cntt - phần mềm     817
cntt - phần cứng    431
an ninh - bảo vệ    152
an ninh              68
Name: count, dtype: int64

In [21]:
pd.set_option("display.max_colwidth", 80)

print("Nhãn 'mạng':")
display(df[df["field_list"].apply(lambda xs: "mạng" in xs)]["job_title"].sample(25, random_state=1))

print("Có nhãn IT nhưng tiêu đề không khớp từ khóa:")
display(df[mask_it & ~mask_title]["job_title"].sample(30, random_state=1))

print("Tiêu đề giống IT nhưng không có nhãn IT:")
display(df[mask_title & ~mask_it]["job_title"].sample(30, random_state=1))

Nhãn 'mạng':


32053                                                 lập trình viên phần mềm điện thoại
36581                                                                 nhân viên kỹ thuật
36582                                                                chuyên viên kế toán
8584                                                                        nhân viên it
28764                               hỗ trợ cơ sở hạ tầng kỹ thuật số công nghệ thông tin
8457                                                           chuyên viên vận hành cntt
279                                                                          tech leader
35866                                                          nhân viên giám sát camera
6755                                                               nhân viên it helpdesk
30867                                    giảng viên quản trị vận hành hệ thống mạng cntt
28563                                                        nhân viên quản trị hệ thống
11098                

Có nhãn IT nhưng tiêu đề không khớp từ khóa:


34878                                                     technical support engineer
48292                                                                       back end
75703                                                            full stack engineer
64038                                                                   tổ trưởng it
73482                                                    nhân viên kỹ thuật phần mềm
71707                                                      network security engineer
65776                                                                account manager
76208                                                                 intern finance
67510                                                                   qa qc tập sự
67612                                               product marketing intern fresher
18708                                           thực tập sinh sale marketing qtscq12
48146                                                            

Tiêu đề giống IT nhưng không có nhãn IT:


75020    nhân viên call center tổng đài viên bán hàng telesales data hot lead thu nhậ...
79461                                                                  android developer
81057                                                                       audit senior
84634                            gia sư luyện thi chứng chỉ it passport online part time
80776                                                     database administration oracle
45291                                                     kĩ sư kinh doanh cung cấp data
8338                                                                 kỹ sư lập trình cam
24919             nhân viên tư vấn gói chăm sóc sức khỏe data có sẵn không đi thị trường
77071                  nhân viên telesales kênh topup xsell thẻ tín dụng hỗ trợ 100 data
75987                                   nhân viên tư vấn telesales có sẵn data tiềm năng
42692                                         it communicator phiên dịch viên tiếng nhật
73805                

In [22]:
STRONG_TITLE = "|".join([
    r"developer", r"lập trình", r"kỹ sư phần mềm", r"software", r"devops",
    r"tester", r"kiểm thử", r"back ?end", r"front ?end", r"full ?stack",
    r"help ?desk", r"\bit\b", r"cntt", r"công nghệ thông tin",
    r"quản trị (hệ thống|mạng|cơ sở dữ liệu)",
    r"data (engineer|analyst|scientist)", r"kỹ sư dữ liệu", r"phân tích dữ liệu",
    r"machine learning", r"\bai\b", r"embedded", r"database",
])

NEGATIVE_TITLE = "|".join([
    r"kinh doanh", r"sales", r"telesales", r"bán hàng", r"kế toán", r"tài chính",
    r"chăm sóc khách hàng", r"tổng đài", r"marketing", r"gia sư", r"giảng viên",
    r"phiên dịch", r"nhập liệu", r"data entry", r"có sẵn data", r"data (tiềm năng|marketing|hot)",
    r"lập trình cam", r"intern finance",
])

title = df["title_clean"] if "title_clean" in df else df["job_title"].fillna("")

strong = title.str.contains(STRONG_TITLE, case=False, regex=True)
neg = title.str.contains(NEGATIVE_TITLE, case=False, regex=True)

mask_it_label = mask_it | mask_data           # từ bước trước
df["is_it"] = (strong | mask_it_label) & ~neg

df["it_source"] = "none"
df.loc[df["is_it"] & strong & mask_it_label, "it_source"] = "both"
df.loc[df["is_it"] & strong & ~mask_it_label, "it_source"] = "title_only"
df.loc[df["is_it"] & ~strong & mask_it_label, "it_source"] = "label_only"

print(df["it_source"].value_counts())
print("Tổng IT:", df["is_it"].sum())

C:\Users\User\AppData\Local\Temp\ipykernel_16692\668742956.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  strong = title.str.contains(STRONG_TITLE, case=False, regex=True)
C:\Users\User\AppData\Local\Temp\ipykernel_16692\668742956.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  neg = title.str.contains(NEGATIVE_TITLE, case=False, regex=True)


it_source
none          82411
both           1314
label_only     1303
title_only      442
Name: count, dtype: int64
Tổng IT: 3059


In [23]:
pd.set_option("display.max_colwidth", 70)
for src in ["both", "label_only", "title_only"]:
    print(f"=== {src} ===")
    display(df[df["it_source"] == src]["job_title"].sample(20, random_state=1).to_frame())

print("=== bị loại bởi từ khóa loại trừ (nhãn/ tiêu đề IT nhưng dính negative) ===")
display(df[(strong | mask_it_label) & neg]["job_title"].sample(20, random_state=1).to_frame())

=== both ===


,job_title
18613,nhân viên it
9490,quản trị hệ thống
26422,kỹ sư lập trình c python ai
51330,java developer
17860,devops
73047,developer full stack
70589,việc làm it comtor tiếng nhật
28502,middle java developer upto 40m
27356,chuyên viên kiểm thử phần mềm
25652,nhân viên it phần mềm thu nhập từ 15 20 triệu


=== label_only ===


,job_title
44040,sale leader mảng phần mềm
77545,senior digital project management specialist
11235,engineer of development center
63582,nhân viên phân tích nghiệp vụ
58006,manual test
19134,nhân viên quản lý dữ liệu tại go bến tre
70242,chuyên viên tư vấn và triển khai phần mềm
61505,giáo viên tin học
77578,3d modeler
57860,graphic designer


=== title_only ===


,job_title
80402,chuyên viên kiểm thử
84897,backend developer
81701,automation tester
11561,kỹ sư embedded firmware engineer
83104,it support
85362,back end developer
81622,android developer
85452,back end developer
81608,android developer
1427,product developer


=== bị loại bởi từ khóa loại trừ (nhãn/ tiêu đề IT nhưng dính negative) ===


,job_title
30251,chuyên viên kinh doanh dự án cntt account manager
75878,kế toán tư vấn triển khai
31182,nhân viên kinh doanh phần mềm quản lý bán hàng bình dương
46252,nhân viên kinh doanh
53434,marketing planner
33202,trợ lý kinh doanh
28274,regional sales manager
75980,gia sư luyện thi chứng chỉ it passport online part time
30019,account manager nhân viên kinh doanh
52760,nhân viên kinh doanh phần mềm


In [24]:
check = pd.concat([
    df[df["it_source"] == "label_only"].sample(40, random_state=7),
    df[df["it_source"] == "both"].sample(10, random_state=7),
    df[df["it_source"] == "title_only"].sample(10, random_state=7),
])[["job_title", "job_fields", "it_source"]].copy()

check["is_it_true"] = ""
check.to_csv(ROOT / "data" / "raw" / "label_check.csv", index=True, encoding="utf-8-sig")

In [25]:
chk = pd.read_csv(ROOT / "data" / "raw" / "label_check_v1_labeled.csv", encoding="utf-8-sig", index_col=0)
chk = chk.dropna(subset=["is_it_true"]).copy()
chk["is_it_true"] = chk["is_it_true"].astype(int)

print("số dòng đã gắn nhãn:", len(chk))
print(chk.groupby("it_source")["is_it_true"].agg(["mean", "count"]))

số dòng đã gắn nhãn: 60
             mean  count
it_source               
both        1.000     10
label_only  0.425     40
title_only  0.900     10


In [26]:
it_df = df[df["is_it"]].copy()
cols = ["job_title", "city", "experience", "salary", "skills"]
print("dòng trùng:", it_df.duplicated(subset=cols).sum())
it_df["job_title"].value_counts().head(15)

dòng trùng: 1502


job_title
nhân viên it                       60
backend developer                  46
business analyst                   36
chuyên viên phân tích dữ liệu      27
android developer                  27
full stack developer               25
nhân viên kỹ thuật                 25
back end developer                 25
automation tester                  24
chuyên viên it                     23
chuyên viên phân tích nghiệp vụ    22
it helpdesk                        21
nhân viên seo web                  21
manual tester                      21
tester                             20
Name: count, dtype: int64

In [27]:
print("thiếu skills:", it_df["skills"].isnull().mean())
display(it_df["skills"].dropna().sample(10, random_state=1).tolist())

sk = it_df["skills"].dropna().str.split(",").explode().str.strip()
sk.value_counts().head(30)

thiếu skills: 0.13141549525988885


['hành chính văn phòng',
 'web developer, lập trình viên, lập trình, chuyên viên lập trình devops engineer, web programmer',
 'embedded system developer kỹ sư lập trình nhúng firmware',
 'test, quản lý chất lượng (qa, qa, qc)',
 'c#, an ninh mạng, .net, phần cứng máy tính, điện thoại',
 'quản trị mạng, it support',
 'c#, python, .net',
 'it helpdesk, it support, quản trị hệ thống, quản trị mạng, quản lý dự án (it)',
 'quản lý dự án (it)',
 'bảo trì sửa chữa, điện thoại, phần cứng máy tính']

skills
                       328
it support             296
sql                    262
java                   258
javascript             245
triển khai phần mềm    226
it phần cứng mạng      223
qa                     208
.net                   201
c#                     195
test                   187
it phần mềm            177
php                    165
quản lý dự án (it)     162
data analytics         160
frontend (html         159
css)                   159
quản trị hệ thống      152
quản trị mạng          142
it helpdesk            137
khoa học kỹ thuật      123
tư vấn bán hàng        119
backend                111
nodejs                 109
phần cứng máy tính     103
điện thoại             103
full stack             101
python                  88
cơ khí                  79
android                 77
Name: count, dtype: int64

In [28]:
test = pd.read_csv(ROOT / "data" / "raw" / "label_check_v1_labeled.csv", encoding="utf-8-sig", index_col=0)
print(len(test), test["is_it_true"].notna().sum())
test.head(3)

60 60


,job_title,job_fields,it_source,is_it_true
24987,nhân viên kỹ thuật hải dương,", bưu chính viễn thông, it phần cứng - mạng, điện - điện tử - điện...",label_only,0
66206,al engineer,"it phần mềm, khoa học, kỹ thuật",label_only,1
26688,nhân viên kiểm duyệt hồ sơ khách hàng sử dụng dịch vụ viettel,"kinh doanh, cntt - phần mềm, xuất bản, bán hàng, in ấn",label_only,0


In [29]:
STRONG_TITLE = "|".join([
    r"developer", r"lập trình", r"kỹ sư phần mềm", r"software", r"devops",
    r"tester", r"kiểm thử", r"back ?end", r"front ?end", r"full ?stack",
    r"help ?desk", r"\bit\b", r"cntt", r"công nghệ thông tin",
    r"quản trị (?:hệ thống|mạng|cơ sở dữ liệu)",
    r"data (?:engineer|analyst|scientist)", r"kỹ sư dữ liệu", r"phân tích dữ liệu",
    r"machine learning", r"\bai\b", r"embedded", r"database",
])

NEGATIVE_TITLE = "|".join([
    r"kinh doanh", r"sales", r"\bsale\b", r"telesales", r"bán hàng", r"kế toán", r"tài chính",
    r"chăm sóc khách hàng", r"customer (?:service|support)", r"tổng đài", r"marketing",
    r"gia sư", r"giảng viên", r"giáo viên", r"phiên dịch", r"nhập liệu", r"data entry",
    r"có sẵn data", r"data (?:tiềm năng|marketing|hot)", r"lập trình cam", r"\bcnc\b",
    r"mạng xã hội", r"\bseo\b", r"designer", r"quay dựng", r"chạy ads", r"bảo trì điện",
    r"xử lý dữ liệu", r"xuất nhập dữ liệu", r"nhập dữ liệu", r"dữ liệu sản xuất",
])

WEAK_TITLE = "|".join([
    r"phân tích nghiệp vụ", r"business analyst", r"ui ?ux", r"security", r"bảo mật",
    r"an ninh mạng", r"an toàn thông tin", r"network", r"\.net\b", r"dataops",
    r"triển khai phần mềm", r"technical support", r"\bai\b", r"máy tính", r"laptop",
    r"\bsre\b", r"brse", r"solution", r"system", r"hệ thống", r"dữ liệu",
])

title = df["job_title"].fillna("")
strong = title.str.contains(STRONG_TITLE, case=False, regex=True)
neg    = title.str.contains(NEGATIVE_TITLE, case=False, regex=True)
weak   = title.str.contains(WEAK_TITLE, case=False, regex=True)

df["is_it"] = (strong & ~neg) | (mask_it_label & weak & ~neg)

df["it_confidence"] = "none"
df.loc[df["is_it"] & strong, "it_confidence"] = "high"
df.loc[df["is_it"] & ~strong, "it_confidence"] = "medium"

print(df["it_confidence"].value_counts())
print("Tổng IT:", df["is_it"].sum())

it_confidence
none      83409
high       1713
medium      348
Name: count, dtype: int64
Tổng IT: 2061


In [30]:
chk["kept"] = df.loc[chk.index, "is_it"]
print(pd.crosstab(chk["kept"], chk["is_it_true"], rownames=["giữ lại"], colnames=["IT thật"]))

IT thật   0   1
giữ lại        
False    22   6
True      2  30


In [31]:
v2 = pd.read_csv(ROOT / "data" / "raw" / "label_check_v2_labeled.csv", encoding="utf-8-sig", index_col=0)
v2 = v2.dropna(subset=["is_it_true"]).copy()
v2["is_it_true"] = v2["is_it_true"].astype(int)
print(len(v2), v2["is_it_true"].mean())

40 0.775


In [32]:
print(df["is_it"].sum())
display(df.loc[[21, 26, 28]].head() if False else df[df["job_title"].str.contains("xử lý dữ liệu|nhập dữ liệu", na=False)]["is_it"].value_counts())

2061


is_it
False    26
Name: count, dtype: int64

In [33]:
df[df["job_title"].str.contains("xử lý dữ liệu|nhập dữ liệu", na=False)]["is_it"].value_counts()

is_it
False    26
Name: count, dtype: int64

In [34]:
it_df = df[df["is_it"]].copy()
cols = ["job_title", "city", "experience", "salary", "skills"]
it_df["is_duplicate"] = it_df.duplicated(subset=cols, keep="first")
it_unique = it_df[~it_df["is_duplicate"]].copy()

print("trước:", len(it_df), "| sau khử trùng:", len(it_unique))
print(it_df["it_confidence"].value_counts())
it_unique["job_title"].value_counts().head(15)

trước: 2061 | sau khử trùng: 1016
it_confidence
high      1713
medium     348
Name: count, dtype: int64


job_title
nhân viên it                       36
backend developer                  22
chuyên viên phân tích dữ liệu      19
business analyst                   18
chuyên viên it                     14
android developer                  14
back end developer                 13
nhân viên it helpdesk              12
it helpdesk                        12
chuyên viên phân tích nghiệp vụ    10
automation tester                   9
tester                              8
data analyst                        7
devops engineer                     7
lập trình viên                      7
Name: count, dtype: int64

In [35]:
import re

def split_skills(s):
    """Tách chuỗi kỹ năng theo dấu phẩy, bỏ qua dấu phẩy nằm trong ngoặc."""
    if pd.isna(s):
        return []
    parts = re.split(r",\s*(?![^()]*\))", s)
    return [p.strip() for p in parts if p.strip()]

SKILL_ALIASES = {
    "python": r"\bpython\b",
    "java": r"\bjava\b(?!\s*script)",
    "javascript": r"javascript|\bjs\b|node\.?js|react(?:js)?\b|angular|vue(?:js)?\b",
    "c#/.net": r"c#|\.net\b|asp\.net",
    "php": r"\bphp\b|laravel",
    "sql": r"\bsql\b|mysql|postgres|oracle|database|cơ sở dữ liệu",
    "frontend": r"front ?end|html|css",
    "backend": r"back ?end",
    "mobile": r"android|\bios\b|flutter|react native|swift|kotlin",
    "testing/qa": r"\btest(?:er|ing)?\b|\bqa\b|\bqc\b|kiểm thử|selenium",
    "devops/cloud": r"devops|docker|kubernetes|\baws\b|azure|\bgcp\b|\bci/?cd\b",
    "data": r"data (?:analy|engineer|scien)|phân tích dữ liệu|power ?bi|tableau|\betl\b|spark|airflow|kafka",
    "ai/ml": r"machine learning|deep learning|\bai\b|\bllm\b|\bnlp\b",
    "it support/network": r"it support|helpdesk|help desk|quản trị (?:mạng|hệ thống)|network|phần cứng máy tính",
    "business analysis": r"business analyst|phân tích nghiệp vụ",
    "security": r"an ninh mạng|bảo mật|security|an toàn thông tin",
}

def extract_skills(text):
    return [name for name, pat in SKILL_ALIASES.items() if re.search(pat, text)]

In [36]:
it_unique["skill_list"] = it_unique["skills"].apply(split_skills)
text = (it_unique["skills"].fillna("") + " " + it_unique["job_title"].fillna("")).str.lower()
it_unique["tech_skills"] = text.apply(extract_skills)

print("có ít nhất 1 kỹ năng:", (it_unique["tech_skills"].str.len() > 0).mean())
print(it_unique["tech_skills"].explode().value_counts())

for name in ["data", "ai/ml", "devops/cloud"]:
    print(name, it_unique["tech_skills"].apply(lambda xs: name in xs).sum())

có ít nhất 1 kỹ năng: 0.7391732283464567
tech_skills
it support/network    201
data                  102
javascript            102
testing/qa             92
backend                88
sql                    88
business analysis      84
java                   69
frontend               67
c#/.net                59
php                    58
mobile                 50
ai/ml                  29
devops/cloud           29
security               27
python                 27
Name: count, dtype: int64
data 102
ai/ml 29
devops/cloud 29
